In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Figure 1.1 生成脚本 (Alexander German Thesis Style)
--------------------------------------------------
复刻目标: Figure 2 from German et al. (NeuroImage 2021)
切片方向: Dim 0 (Axial)
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
import h5py
import os

# ================= 核心配置 (修改这里) =================

# 1. 输入文件路径
# DATA_PATH = "/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal/your_patient_data.mat"

DATA_PATH = "/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal/ODP_01_qhlazec_3d_validated_minimal.mat"


# 2. Saliency 数据路径 (没有则填 None，自动生成模拟数据)
SALIENCY_PATH = None 

# 3. 切片选择 (Dim 0) -> 您的 Axial 方向
SLICE_IDX = 222  # 请根据 Dim 0 的范围调整 (通常是 X 轴的长度)

# 4. 选点配置 (Dim0, Dim1, Dim2) -> None 表示自动选 Mask 中心
VOXEL_COORDS = None 

# 5. 输出文件名
OUTPUT_FILENAME = "Figure_1_1_AlexStyle_Dim0.png"

# ================= 模态映射与配置 =================

# 0-based 映射 (参考您的 1-based 描述)
# 1-15 QTI -> 0-14
# 16-225 b-tensor -> 15-224 (lin:15-95, plan:96-176, spher:177-224)
# 226-229 CEST -> 225-228
# 230 M0 -> 229
# ...

# 1. Alex 选中的 8 张代表性切片
# 注意：这里我是根据通用顺序推断的 Index，您可能需要根据具体数据微调
ALEX_SLICES = [
    {"idx": 10,  "label": "MD",       "type": "metric"}, # 假设 MD 在第11个
    {"idx": 14,  "label": "µFA",      "type": "metric"}, # 假设 uFA 在第15个
    {"idx": 55,  "label": "lin",      "type": "raw"},    # 线性 b-tensor 中间
    {"idx": 135, "label": "plan",     "type": "raw"},    # 平面 b-tensor 中间
    {"idx": 200, "label": "spher",    "type": "raw"},    # 球形 b-tensor 中间
    {"idx": 227, "label": "NOE",      "type": "fit"},    # NOE fit
    {"idx": 229, "label": "-300 ppm", "type": "raw"},    # M0 / Ref
    {"idx": 300, "label": "-4 ppm",   "type": "raw"},    # Z-spec 里的某一点
]

# 2. 底部曲线的背景分区 (参考原图的灰色背景)
# 原图里 QTI Raw 区域是灰底，CEST Raw 也是灰底
BG_SHADES = [
    {"start": 15,  "end": 225, "color": "#F0F0F0", "label": None}, # b-tensors
    {"start": 230, "end": 341, "color": "#F0F0F0", "label": None}, # Z-spectra (High B1 part mostly)
]

# 3. 底部曲线的文字标注 (x坐标, 文字)
TEXT_ANNOTATIONS = [
    (5,   "µFA",       "bottom"),
    (55,  "linear",    "top"),
    (135, "planar",    "top"),
    (200, "spherical", "top"),
    (227, "MT\n-4ppm", "bottom"),
    (290, "-300 ppm\n-4 ppm", "bottom"), # 近似位置
]

# 大标题标注
SECTION_LABELS = [
    (15,  "QTI"),
    (120, "b-tensors"),
    (220, "CEST"),
    (280, "Z-spectra")
]

# ================= 辅助函数 =================

def load_data(mat_path):
    print(f"Loading: {mat_path} ...")
    with h5py.File(mat_path, 'r') as f:
        data = f['data'][:]
        if data.shape[0] in [341, 351]:
            data = np.moveaxis(data, 0, -1)
        region_mask = f['region_mask'][:].astype(bool)
    return data, region_mask

def normalize_image(img):
    img = img.astype(np.float32)
    p1, p99 = np.percentile(img, [1, 99])
    return np.clip((img - p1) / (p99 - p1 + 1e-8), 0, 1)

def get_saliency_vector(length=341):
    """模拟 Saliency 曲线 (如果没有真实数据)"""
    if SALIENCY_PATH and os.path.exists(SALIENCY_PATH):
        sal = np.load(SALIENCY_PATH)
        if len(sal) >= length: return sal[:length]
        else: return np.pad(sal, (0, length-len(sal)))
    else:
        # 模拟一个像图里那样的红色曲线：低底噪，偶尔有尖峰
        np.random.seed(10)
        x = np.linspace(0, length, length)
        sal = np.abs(np.random.randn(length)) * 0.05
        # 在关键点加峰
        sal[6] += 0.4   # FA
        sal[14] += 0.3  # uFA
        sal[55] += 0.2  # lin
        sal[227] += 0.8 # NOE/MT area
        sal[300] += 0.5 # Z-spec dip
        return sal

def get_auto_coordinates(mask):
    indices = np.argwhere(mask)
    if len(indices) == 0: return tuple(np.array(mask.shape)//2)
    center = indices.mean(axis=0).astype(int)
    dists = np.sum((indices - center)**2, axis=1)
    return tuple(indices[np.argmin(dists)])

# ================= 主绘图逻辑 =================

def plot_alex_style(data, mask, voxel_3d, output_filename):
    v0, v1, v2 = voxel_3d
    
    # === Dim 0 切片逻辑 ===
    # 固定 Dim 0 = v0
    # 图像平面是 (Dim 1, Dim 2) -> (Y, Z)
    # Scatter 坐标: (v2, v1) -> (X=Z, Y=Y)
    
    # 检查切片索引是否越界
    if v0 >= data.shape[0]:
        v0 = data.shape[0] // 2
        print(f"[Warn] Slice index adjusted to {v0}")

    # 提取曲线数据
    intensity_sig = data[v0, v1, v2, :341].astype(np.float32)
    saliency_sig = get_saliency_vector(341)

    # 准备画布
    fig = plt.figure(figsize=(16, 10), facecolor='white')
    # 上下两部分：上面放切片，下面放曲线
    gs = gridspec.GridSpec(2, 8, height_ratios=[1, 2.5], hspace=0.15, wspace=0.05)

    # --- PART 1: Top Row Images (8 Slices) ---
    for i, item in enumerate(ALEX_SLICES):
        ax = fig.add_subplot(gs[0, i])
        ch_idx = item['idx']
        
        # 提取切片: Dim 0 固定
        slc = data[v0, :, :, ch_idx]
        slc_norm = normalize_image(slc)
        
        # 显示
        ax.imshow(slc_norm, cmap='gray', aspect='equal', origin='upper')
        
        # 标记点 (Alex 图里是一个白色/红色的小方框，这里用红色方框)
        # 坐标: scatter(x, y) -> (col, row) -> (v2, v1)
        rect_size = 5
        rect = patches.Rectangle((v2 - rect_size/2, v1 - rect_size/2), 
                                 rect_size, rect_size, 
                                 linewidth=1.5, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        
        # 标签 (放在下方，白色字体，模仿原图)
        ax.text(0.5, 0.05, item['label'], transform=ax.transAxes, 
                color='white', fontsize=11, fontweight='bold', ha='center', va='bottom')
        
        ax.axis('off')

    # --- PART 2: Bottom Graph (Dual Axis) ---
    ax_main = fig.add_subplot(gs[1, :]) # 跨越所有列
    ax_sal = ax_main.twinx()

    # 1. 绘制背景区域 (Shading)
    for bg in BG_SHADES:
        ax_main.axvspan(bg['start'], bg['end'], color=bg['color'], alpha=1.0, zorder=0)

    # 2. 绘制 Intensity (左轴，黑色)
    ax_main.plot(intensity_sig, color='black', linewidth=1.0, zorder=2)
    ax_main.set_ylabel("voxel-intensity", fontsize=12, color='black')
    ax_main.set_ylim(0, np.max(intensity_sig) * 1.1)
    
    # 3. 绘制 Saliency (右轴，红色)
    ax_sal.plot(saliency_sig, color='#C00000', linewidth=0.8, zorder=3)
    # 红色填充
    ax_sal.fill_between(range(len(saliency_sig)), 0, saliency_sig, color='#FF0000', alpha=0.1, zorder=3)
    ax_sal.set_ylabel("saliency", fontsize=12, color='#C00000')
    ax_sal.set_ylim(0, np.max(saliency_sig) * 1.5) # 让红色曲线稍微矮一点，不喧宾夺主
    ax_sal.tick_params(axis='y', colors='#C00000')

    # 4. 添加垂直分割线
    lines = [15, 96, 177, 225, 230]
    for x in lines:
        ax_main.axvline(x, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)

    # 5. 添加文字标注 (Annotations)
    # 区域大标题
    for x, text in SECTION_LABELS:
        # 放在图的最上方
        ax_main.text(x, ax_main.get_ylim()[1]*0.95, text, 
                     ha='left', fontsize=12, color='#333333', zorder=10)

    # 局部标注 (linear, planar 等)
    for x, text, pos in TEXT_ANNOTATIONS:
        if pos == 'top':
            y_pos = ax_main.get_ylim()[1] * 0.3 # 放在 Intensity 曲线下方一点的空白处
        else:
            y_pos = 0 # 放在x轴附近
        
        ax_main.text(x, y_pos, text, ha='left', va='bottom', fontsize=10, color='#333333', rotation=0)

    # 6. 模仿原图的斜向文字 (100 s/mm^2 等)
    # 这里我们只是静态标注，不做复杂的 tick logic
    # ax_main.text(70,  ax_main.get_ylim()[1]*0.1, "linear", ha='center', fontsize=10)
    # ax_main.text(140, ax_main.get_ylim()[1]*0.1, "planar", ha='center', fontsize=10)
    # ax_main.text(200, ax_main.get_ylim()[1]*0.2, "spherical", ha='center', fontsize=10)
    ax_main.text(250, ax_main.get_ylim()[1]*0.4, "0.7 µT", ha='center', fontsize=10)
    ax_main.text(310, ax_main.get_ylim()[1]*0.4, "1.0 µT", ha='center', fontsize=10)

    # 样式微调
    ax_main.set_xlim(0, 341)
    ax_main.spines['top'].set_visible(False)
    ax_sal.spines['top'].set_visible(False)
    
    # 底部说明文字
    plt.figtext(0.15, 0.02, "neural network classifies single voxel spectrum", fontsize=11, ha='left')
    plt.figtext(0.85, 0.02, "segmentation of each individual voxel", fontsize=11, ha='right')
    
    # 标题
    plt.suptitle("Morphological, chemically and microstructurally weighted images", fontsize=14, y=0.96)

    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Generated: {output_filename}")


def main():
    # 1. 加载
    data, mask = load_data(DATA_PATH)
    
    # 2. 选点
    if VOXEL_COORDS is not None:
        voxel = VOXEL_COORDS
    else:
        # 这里稍微复杂点：我们需要在 Dim 0 = SLICE_IDX 的平面上找点
        # 先切片 Mask
        if SLICE_IDX < mask.shape[0]:
            mask_slice = mask[SLICE_IDX, :, :]
            indices = np.argwhere(mask_slice)
            if len(indices) > 0:
                # 找切片重心
                center = indices.mean(axis=0).astype(int)
                # v1=center[0], v2=center[1]
                voxel = (SLICE_IDX, center[0], center[1])
            else:
                voxel = tuple(np.array(mask.shape)//2)
        else:
             voxel = tuple(np.array(mask.shape)//2)
             
    print(f"Selected Voxel: {voxel} (Dim0={voxel[0]} is the slice)")

    # 3. 绘图
    plot_alex_style(data, mask, voxel, OUTPUT_FILENAME)

if __name__ == "__main__":
    main()

In [ ]:
#旋转180度

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
import h5py
import os

# ================= 核心配置 =================

# 1. 输入文件路径 (请确保在 Notebook 环境中路径正确)
DATA_PATH = "/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal/ODP_01_qhlazec_3d_validated_minimal.mat"

# 2. Saliency 数据路径
SALIENCY_PATH = None 

# 3. 切片选择
SLICE_IDX = 222 

# 4. 选点配置
VOXEL_COORDS = None 

# 5. Alex 选中的 8 张代表性切片配置
ALEX_SLICES = [
    {"idx": 10,  "label": "MD",       "type": "metric"},
    {"idx": 14,  "label": "µFA",      "type": "metric"},
    {"idx": 55,  "label": "lin",      "type": "raw"},
    {"idx": 135, "label": "plan",     "type": "raw"},
    {"idx": 200, "label": "spher",    "type": "raw"},
    {"idx": 227, "label": "NOE",      "type": "fit"},
    {"idx": 229, "label": "-300 ppm", "type": "raw"},
    {"idx": 300, "label": "-4 ppm",   "type": "raw"},
]

BG_SHADES = [
    {"start": 15,  "end": 225, "color": "#F0F0F0", "label": None},
    {"start": 230, "end": 341, "color": "#F0F0F0", "label": None},
]

TEXT_ANNOTATIONS = [
    (5,   "µFA",       "bottom"),
    (55,  "linear",    "top"),
    (135, "planar",    "top"),
    (200, "spherical", "top"),
    (227, "MT\n-4ppm", "bottom"),
    (290, "-300 ppm\n-4 ppm", "bottom"),
]

SECTION_LABELS = [
    (15,  "QTI"),
    (120, "b-tensors"),
    (220, "CEST"),
    (280, "Z-spectra")
]

# ================= 辅助函数 =================

def load_data(mat_path):
    print(f"Loading: {mat_path} ...")
    # 增加容错，防止在没有该路径的环境报错
    if not os.path.exists(mat_path):
        print(f"Error: File not found at {mat_path}")
        return None, None
        
    with h5py.File(mat_path, 'r') as f:
        data = f['data'][:]
        if data.shape[0] in [341, 351]:
            data = np.moveaxis(data, 0, -1)
        region_mask = f['region_mask'][:].astype(bool)
    return data, region_mask

def normalize_image(img):
    img = img.astype(np.float32)
    p1, p99 = np.percentile(img, [1, 99])
    return np.clip((img - p1) / (p99 - p1 + 1e-8), 0, 1)

def get_saliency_vector(length=341):
    if SALIENCY_PATH and os.path.exists(SALIENCY_PATH):
        sal = np.load(SALIENCY_PATH)
        if len(sal) >= length: return sal[:length]
        else: return np.pad(sal, (0, length-len(sal)))
    else:
        np.random.seed(10)
        x = np.linspace(0, length, length)
        sal = np.abs(np.random.randn(length)) * 0.05
        sal[6] += 0.4; sal[14] += 0.3; sal[55] += 0.2
        sal[227] += 0.8; sal[300] += 0.5
        return sal

# ================= 主绘图逻辑 =================

def plot_alex_style_notebook(data, voxel_3d):
    """
    专为 Jupyter Notebook 设计的绘图函数
    """
    v0, v1, v2 = voxel_3d
    
    if v0 >= data.shape[0]: v0 = data.shape[0] // 2

    intensity_sig = data[v0, v1, v2, :341].astype(np.float32)
    saliency_sig = get_saliency_vector(341)

    # 创建画布
    fig = plt.figure(figsize=(16, 10), facecolor='white')
    gs = gridspec.GridSpec(2, 8, height_ratios=[1, 2.5], hspace=0.15, wspace=0.05)

    # --- PART 1: Top Row Images (8 Slices) ---
    for i, item in enumerate(ALEX_SLICES):
        ax = fig.add_subplot(gs[0, i])
        ch_idx = item['idx']
        
        # 1. 提取
        slc = data[v0, :, :, ch_idx]
        
        # 2. 旋转 180 度 (k=2)
        slc_rotated = np.rot90(slc, k=2)
        slc_norm = normalize_image(slc_rotated)
        
        ax.imshow(slc_norm, cmap='gray', aspect='equal', origin='upper')
        
        # 3. 坐标修正 (旋转180度后的坐标)
        H, W = slc.shape
        new_v1 = H - 1 - v1 # Row
        new_v2 = W - 1 - v2 # Col
        
        rect_size = 5
        rect = patches.Rectangle((new_v2 - rect_size/2, new_v1 - rect_size/2), 
                                 rect_size, rect_size, 
                                 linewidth=1.5, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        
        ax.text(0.5, 0.05, item['label'], transform=ax.transAxes, 
                color='white', fontsize=11, fontweight='bold', ha='center', va='bottom')
        ax.axis('off')

    # --- PART 2: Bottom Graph ---
    ax_main = fig.add_subplot(gs[1, :])
    ax_sal = ax_main.twinx()

    for bg in BG_SHADES:
        ax_main.axvspan(bg['start'], bg['end'], color=bg['color'], alpha=1.0, zorder=0)

    ax_main.plot(intensity_sig, color='black', linewidth=1.0, zorder=2)
    ax_main.set_ylabel("voxel-intensity", fontsize=12, color='black')
    ax_main.set_ylim(0, np.max(intensity_sig) * 1.1)
    
    ax_sal.plot(saliency_sig, color='#C00000', linewidth=0.8, zorder=3)
    ax_sal.fill_between(range(len(saliency_sig)), 0, saliency_sig, color='#FF0000', alpha=0.1, zorder=3)
    ax_sal.set_ylabel("saliency", fontsize=12, color='#C00000')
    ax_sal.set_ylim(0, np.max(saliency_sig) * 1.5)
    ax_sal.tick_params(axis='y', colors='#C00000')

    lines = [15, 96, 177, 225, 230]
    for x in lines:
        ax_main.axvline(x, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)

    for x, text in SECTION_LABELS:
        ax_main.text(x, ax_main.get_ylim()[1]*0.95, text, 
                     ha='left', fontsize=12, color='#333333', zorder=10)

    for x, text, pos in TEXT_ANNOTATIONS:
        y_pos = ax_main.get_ylim()[1] * 0.3 if pos == 'top' else 0
        ax_main.text(x, y_pos, text, ha='left', va='bottom', fontsize=10, color='#333333')

    ax_main.text(250, ax_main.get_ylim()[1]*0.4, "0.7 µT", ha='center', fontsize=10)
    ax_main.text(310, ax_main.get_ylim()[1]*0.4, "1.0 µT", ha='center', fontsize=10)

    ax_main.set_xlim(0, 341)
    ax_main.spines['top'].set_visible(False)
    ax_sal.spines['top'].set_visible(False)
    
    plt.figtext(0.15, 0.02, "neural network classifies single voxel spectrum", fontsize=11, ha='left')
    plt.figtext(0.85, 0.02, "segmentation of each individual voxel", fontsize=11, ha='right')
    plt.suptitle("Morphological, chemically and microstructurally weighted images", fontsize=14, y=0.96)

    # 【关键修改】这里不再 savefig，而是直接 show
    plt.show()

# ================= 执行逻辑 =================

# 1. 加载数据
data, mask = load_data(DATA_PATH)

if data is not None:
    # 2. 确定坐标
    if VOXEL_COORDS is not None:
        voxel = VOXEL_COORDS
    else:
        if SLICE_IDX < mask.shape[0]:
            mask_slice = mask[SLICE_IDX, :, :]
            indices = np.argwhere(mask_slice)
            if len(indices) > 0:
                center = indices.mean(axis=0).astype(int)
                voxel = (SLICE_IDX, center[0], center[1])
            else:
                voxel = tuple(np.array(mask.shape)//2)
        else:
             voxel = tuple(np.array(mask.shape)//2)
    
    print(f"Selected Voxel: {voxel}")

    # 3. 绘图 (直接在 Notebook 显示)
    plot_alex_style_notebook(data, voxel)
else:
    print("Data load failed, please check DATA_PATH.")